# **Sprint 1: Automated Data Pipeline & Forensic Log Analyzer**

## Project Mission

You are working as an Analytics Engineer / Data Analyst. Microservices generate daily server logs in `data/daily_logs/`. These files arrive in a mix of `.txt` and `.json` formats, and several files are corrupted.

If an automated data pipeline crashes on corrupted data, morning business dashboards fail. Your goal is to build an automated, crash-proof pipeline that filters junk files, safely parses data with Object-Oriented Programming (OOP), aggregates forensic server errors (5xx), consolidates clean data into `analytics_ready.json`, and quarantines corrupted files into `data/quarantine/`.



## Pipeline Architecture Flow

```text
data/daily_logs/
        │
        ▼
┌─────────────────────────────┐
│ Step 1: is_valid_logfile()  │
│ Filter hidden/junk files    │
└──────────────┬──────────────┘
               │
               ▼
      Candidate Log Files
         (72 files)
               │
               ▼
┌─────────────────────────────┐
│ Step 2: DataParser          │
│ Try-Except Validation       │
└──────────────┬──────────────┘
               │
        ┌──────┴──────┐
        ▼             ▼
  Valid Records   Corrupted Files
        │             │
        ▼             ▼
 clean_records   quarantine_list
        │             │
        ▼             ▼
 Step 3: 5xx      Step 4: copy
 Counter          to quarantine/
        │
        ▼
 error_counts
        │
        ▼
analytics_ready.json

In [ ]:
import requests, os

url = "https://api.github.com/repos/JayanGupta/Data_Science_Notes/contents/Sprint%201/data/daily_logs"

os.makedirs("daily_log", exist_ok=True)

for f in requests.get(url).json():
    open("daily_log/" + f["name"], "wb").write(
        requests.get(f["download_url"]).content
    )

Q1: How do we scan a directory and filter out hidden/junk files?


In [33]:
import os

LOG_DIR = "/content/daily_log"

def is_valid_logfiles(filename):
    if filename.startswith(".") or filename.startswith("~"):
        return False

    return filename.lower().endswith((".txt", ".json"))

In [34]:
all = os.listdir(LOG_DIR)

data_files = [os.path.join(LOG_DIR, f) for f in all if is_valid_logfiles(f)]

data_files.sort()

print("Total number of files:", len(all))
print("Total number of valid files:", len(data_files))

Total number of files: 74
Total number of valid files: 72


In [36]:
print(sample)
print(open(sample, "r", encoding="utf-8").read())

/content/daily_log/event_000.json
{
  "timestamp": "2026-03-01T21:49:00",
  "ip_address": "203.0.113.55",
  "status_code": 200,
  "endpoint": "/api/users",
  "response_time_ms": 262
}


In [42]:
import json

with open(sample, "r", encoding="utf-8") as f:
    data = json.load(f)

print(data)

{'timestamp': '2026-03-01T21:49:00', 'ip_address': '203.0.113.55', 'status_code': 200, 'endpoint': '/api/users', 'response_time_ms': 262}


Q2: How do we safely parse a single JSON log file ?


In [65]:
import json

def parse_json_file(filepath):
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)

        record = {
            "source": filepath,
            "ip": data["ip_address"],
            "status": data["status_code"]
        }

        return record

    except (json.JSONDecodeError, KeyError) as e:

        return None , f"{type(e).__name__}:{e}"

In [71]:
sample = [f for f in data_files if f.endswith(".json")][0]

record = parse_json_file(sample)

print(record)

{'source': '/content/daily_log/event_000.json', 'ip': '203.0.113.55', 'status': 200}


In [70]:
record = parse_json_file(sample)
print(record)

(None, 'JSONDecodeError:Extra data: line 1 column 8 (char 7)')


Q3: Do we simply parse a single TXT log file ?

In [76]:
def parse_text_file(filepath):
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            line = f.readline().strip()

        ip, status = line.split(",")

        record = {
            "source": filepath,
            "ip": ip.strip(),
            "status": status.strip()
        }

        return record, None

    except ValueError as e:
        return None, f"{type(e).__name__}:{e}"

In [69]:
sample = [f for f in data_files if f.endswith(".txt")][0]
record= parse_text_file(sample)
print(record)

({'source': '/content/daily_log/access_040.txt', 'ip': '192.168.1.14', 'status': '503'}, None)


Q4: How do we encapsulate this into a clean DataParser calss(OOP)?

In [67]:
class DataParser:

    def __init__(self, filepath):
        self.filepath = filepath
        self.is_valid = True
        self.record = None
        self.error = None

    def parse(self):

        if self.filepath.lower().endswith(".json"):
            self.record, self.error = parse_json_file(self.filepath)

        else:
            self.record, self.error = parse_text_file(self.filepath)

        if self.error is not None:
            self.is_valid = False

        return self.record

In [78]:
# We patch parse_json_file temporarily to return (record, None) when successful
original_parse_json = parse_json_file
def fixed_parse_json_file(filepath):
    result = original_parse_json(filepath)
    if isinstance(result, tuple):
        return result
    return result, None

# Apply patch to the global namespace
globals()["parse_json_file"] = fixed_parse_json_file

bad = 0

for path in data_files:
    parser = DataParser(path)
    parser.parse()

    if not parser.is_valid:
        bad += 1

print("Bad files:", bad)

Bad files: 7


Q5: How do we process all files and aggregate 5xx forensic anomalies?


In [82]:
from collections import Counter
clean_records=[]
bad_records=[]
error_counters= Counter()
for path in data_files:

  parser = DataParser(path)
  parser.parse()

  if parser.is_valid:
    clean_records.append(parser.record)
    if str(parser.record["status"]).startswith("5"):
      error_counters[parser.record["status"]] += 1

  else:
    bad_records.append(path)

In [84]:
len(clean_records)

65

In [83]:
len(bad_records)

7

In [86]:
for ip, count in error_counters.most_common(5):
  print(ip, "" , count)

502  4
500  4
503  3
503  3
502  2


Q6: How do we save the clean data and isolate the corrupt files?

In [88]:
import json
import os
import shutil

OUTPUT_JSON = "analytics_ready.json"
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(clean_records, f, indent=2)

QUARANTINE_DIR = "data/quarantine"
os.makedirs(QUARANTINE_DIR, exist_ok=True)

for path in bad_records:
    des = os.path.join(QUARANTINE_DIR, os.path.basename(path))
    shutil.copy(path, des)